# Knowledge Based Route Finding Agent

## Step 1: Task and environment
The agent finds routes in a directed Kenyan road network and answers route, reachability, region, neighbour and route-listing questions. It uses explicit road, region and closure facts. It is goal-driven and minimizes distance.

## Step 2: Vocabulary and PEAS
**PEAS**: 1.Performance = shortest distance and correct answers 2.Environment = directed roads and towns 3.Actuators = route answers and Knowledge Base updates 4.Sensors = user questions and facts.

**Vocabulary**:
- `Road(a, b, km)`: directed road from `a` to `b` of `km` kilometres.
- `In(town, region)`: town belongs to region.
- `Blocked(a, b)`: the directed road is unavailable.

No API or web scraping has been used.

## Step 3: Initial knowledge base
** 1.`graph` stores roads
** 2.`town_regions` stores region facts 
** 3.`blocked_roads` stores closure facts.

In [ ]:
import heapq
from itertools import count

graph = {
 "Nairobi": [("Nakuru",163),("Narok",145)],
 "Nakuru": [("Kericho",107),("Eldoret",156)],
 "Narok": [("Bomet",81)],
 "Bomet": [("Kericho",76)],
 "Kericho": [("Kisumu",84)],
 "Eldoret": [("Kisumu",138)],
 "Kisumu": []
}
town_regions = {
 "Nairobi":"Nairobi Metropolitan", "Nakuru":"Rift Valley",
 "Narok":"Rift Valley", "Bomet":"Rift Valley",
 "Kericho":"Rift Valley", "Eldoret":"Rift Valley",
 "Kisumu":"Lake Basin"
}
blocked_roads = set()
towns = set(graph)
print("Initial KB:", len(towns), "towns and", sum(map(len, graph.values())), "roads")

## Step 4: Knowledge base rules
- A road is usable unless `Blocked(a,b)` exists
- `CanReach(a,b)` means an open directed path exists
- `Shortest(a,b)` is the minimum-cost open-road path
- Negative distances are invalid

In [ ]:
def add_town(name, region=None):
    name=name.strip().title()
    if not name:
        return False, "Town name cannot be empty."
    if name in graph:
        return False, f"Town {name} already exists."
    graph[name]=[]
    towns.add(name)
    if region:
        town_regions[name]=region.strip().title()
    return True, f"Added town {name}."

def add_road(source,destination,km):
    source,destination=source.strip().title(),destination.strip().title()
    if source not in towns or destination not in towns:
        return False, "Both towns must already exist."
    if not isinstance(km,(int,float)) or km < 0:
        return False, "Distance must be zero or greater."
    graph[source]=[(n,d) for n,d in graph[source] if n != destination]+[(destination,km)]
    return True, f"Added road {source} -> {destination}: {km} km."

def block_road(source,destination):
    source,destination=source.strip().title(),destination.strip().title()
    if source not in towns or destination not in towns:
        return False, "Unknown town in road closure."
    blocked_roads.add((source,destination))
    return True, f"Blocked {source} -> {destination}."

def reopen_road(source,destination):
    source,destination=source.strip().title(),destination.strip().title()
    blocked_roads.discard((source,destination))
    return True, f"Reopened {source} -> {destination}."

## Step 5: Uniform-cost search and knowledge-change demonstration
The heap stores `(cost, insertion_order, town, path)`. `best_cost` holds the cheapest known cost to each town. The stale-entry check ignores an older queue entry when it differs from `best_cost`. The goal is checked at pop, not discovery, which guarantees optimality.

In [ ]:
def uniform_cost_search(start,goal,trace=False):
    if start not in towns or goal not in towns:
        return None, float("inf")
    if start == goal:
        return [start], 0
    sequence=count()
    frontier=[(0,next(sequence),start,[start])]
    best_cost={start:0}
    explored=set()
    while frontier:
        cost,_,current,path=heapq.heappop(frontier)
        if cost != best_cost.get(current):
            continue
        if current in explored:
            continue
        explored.add(current)
        if trace:
            print(f"Expand {current:<10} cost={cost} km")
        if current == goal:
            return path,cost
        for neighbour,edge_cost in graph.get(current,[]):
            if (current,neighbour) in blocked_roads:
                continue
            new_cost=cost+edge_cost
            if neighbour not in explored and new_cost < best_cost.get(neighbour,float("inf")):
                best_cost[neighbour]=new_cost
                heapq.heappush(frontier,(new_cost,next(sequence),neighbour,path+[neighbour]))
    return None,float("inf")

print("Before closure:")
print(uniform_cost_search("Nairobi","Kisumu",True))
print(block_road("Nairobi","Nakuru"))
print("After closing Nairobi -> Nakuru:")
print(uniform_cost_search("Nairobi","Kisumu",True))
print(reopen_road("Nairobi","Nakuru"))

## Step 6: Route and connectivity queries

In [ ]:
def shortest_route(start,goal):
    start,goal=start.strip().title(),goal.strip().title()
    if start not in towns:
        return f"Unknown start town: {start}"
    if goal not in towns:
        return f"Unknown destination town: {goal}"
    if start == goal:
        return f"Start and destination are both {start}. Route cost: 0 km"
    path,cost=uniform_cost_search(start,goal)
    if path is None:
        return f"No open route from {start} to {goal}."
    return f"Shortest route: {' -> '.join(path)} ({cost} km)"

def can_reach(start,goal):
    start,goal=start.strip().title(),goal.strip().title()
    if start not in towns:
        return f"Unknown start town: {start}"
    if goal not in towns:
        return f"Unknown destination town: {goal}"
    path,_=uniform_cost_search(start,goal)
    return f"Yes, {goal} is reachable from {start}." if path else f"No, {goal} is not reachable from {start}."

print(shortest_route("Nairobi","Kisumu"))
print(can_reach("Kisumu","Nairobi"))

## Step 7: Regional and direct-neighbour queries

In [ ]:
def towns_in(region):
    region=region.strip().title()
    return sorted(t for t,r in town_regions.items() if r==region)

def direct_connections(town):
    town=town.strip().title()
    if town not in towns:
        return f"Unknown town: {town}"
    return [(n,k) for n,k in graph[town] if (town,n) not in blocked_roads]

print("Rift Valley:",towns_in("Rift Valley"))
print("Lake Basin:",towns_in("Lake Basin"))
print("Direct from Nakuru:",direct_connections("Nakuru"))

## Step 8: List all routes
This Depth First Search enumerates routes for comparison; UCS is the shortest-route search.

In [ ]:
def all_routes(start,goal):
    start,goal=start.strip().title(),goal.strip().title()
    if start not in towns:
        return f"Unknown start town: {start}",[]
    if goal not in towns:
        return f"Unknown destination town: {goal}",[]
    routes=[]
    def visit(current,path,cost):
        if current == goal:
            routes.append((path,cost)); return
        for neighbour,km in graph.get(current,[]):
            if neighbour not in path and (current,neighbour) not in blocked_roads:
                visit(neighbour,path+[neighbour],cost+km)
    visit(start,[start],0)
    routes.sort(key=lambda item:item[1])
    return f"Routes from {start} to {goal}",routes

title,routes=all_routes("Nairobi","Kisumu")
print(title)
for path,cost in routes:
    print(" -> ".join(path),"=",cost,"km")

## Step 9: Debugging and bad-input handling

In [ ]:
print(shortest_route("Mombasa","Kisumu"))
print(shortest_route("Nairobi","Mombasa"))
print(shortest_route("Nairobi","Nairobi"))
print(shortest_route("Kisumu","Nairobi"))
print(block_road("Nairobi","Nakuru"))
print(shortest_route("Nairobi","Kisumu"))
print(reopen_road("Nairobi","Nakuru"))
print(add_road("Nairobi","Kisumu",-10))

## Step 10: Reflection
Blocking `Nairobi to Nakuru` changes the shortest route from `Nairobi to Nakuru to Kericho to Kisumu` at 354 km to `Nairobi to Narok to Bomet to Kericho to Kisumu` at 386 km.

`best_cost` prevents inferior routes from replacing cheaper costs. The stale-entry check ignores obsolete queue entries. Checking the goal only after pop gives UCS its optimality guarantee. The agent now has explicit facts, update functions, search logic, multiple query types, bad-input checks, and a closure demonstration.

## Interactive menu
Run `run_menu()` once to choose a query. The menu can add, block, reopen, and answer questions.

In [1]:
def run_menu():
    print("1 Shortest route")
    print("2 Can I reach a town?")
    print("3 Towns in a region")
    print("4 Direct connections")
    print("5 List all routes")
    print("6 Block road")
    print("7 Reopen road")
    choice=input("Choose 1-7, or Enter to stop: ").strip()
    if choice == "":
        return
    if choice == "1":
        print(shortest_route(input("Start: "), input("Destination: ")))
    elif choice == "2":
        print(can_reach(input("Start: "), input("Destination: ")))
    elif choice == "3":
        print(towns_in(input("Region: ")))
    elif choice == "4":
        print(direct_connections(input("Town: ")))
    elif choice == "5":
        title,routes=all_routes(input("Start: "), input("Destination: "))
        print(title)
        for path,cost in routes:
            print(" -> ".join(path), "=", cost, "km")
    elif choice == "6":
        print(block_road(input("Source: "), input("Destination: ")))
    elif choice == "7":
        print(reopen_road(input("Source: "), input("Destination: ")))
    else:
        print("Invalid menu choice.")

# run_menu()
